<a href="https://colab.research.google.com/github/smcgovern-proj/neuromatch_ecog_project/blob/main/broadband_mean_electrode.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# broadband_mean_electrode
Get mean power per electrodes per stimulus.

Returns: NxM (N number of stimulus, M mean per electrode)

**Work to be done:**
1. Check what should be the exact time window.
2. Reliably get number of trials and when each start and end for a given subject.

**Possible issues:**
1. Most electrodes are irrelevant to the decision, so we will have lots of noise. We will probably need to do an electrode selection before actually getting mean power.
2. Getting only the mean power across the time stimulus will probably be too little data. Perhaps, we should make more time windows inside a trial and also use those as features (considering we will end up with less electrodes after reduction),

In [3]:
# @title Data retrieval
import os, requests

fname = 'faceshouses.npz'
url = "https://osf.io/argh7/download"

if not os.path.isfile(fname):
  try:
    r = requests.get(url)
  except requests.ConnectionError:
    print("!!! Failed to download data !!!")
  else:
    if r.status_code != requests.codes.ok:
      print("!!! Failed to download data !!!")
    else:
      with open(fname, "wb") as fid:
        fid.write(r.content)

In [ ]:
# @title Imports

from matplotlib import rcParams
from matplotlib import pyplot as plt
rcParams['figure.figsize'] = [20, 4]
rcParams['font.size'] = 15
rcParams['axes.spines.top'] = False
rcParams['axes.spines.right'] = False
rcParams['figure.autolayout'] = True

In [ ]:
# @title Data loading
import numpy as np

alldat = np.load(fname, allow_pickle=True)['dat']

dict_keys(['t_off', 'stim_id', 't_on', 'srate', 'V', 'scale_uv', 'locs', 'hemisphere', 'lobe', 'gyrus', 'Brodmann_Area'])
dict_keys(['stim_id', 'stim_cat', 'stim_noise', 't_on', 't_off', 'key_press', 'V', 'categories', 'scale_uv', 'locs', 'hemisphere', 'lobe', 'gyrus', 'Brodmann_Area'])


In [ ]:
# @title Configure

subject_index = 1 #1 and 2 missing data on dat2

dat1 = alldat[subject_index][0]
dat2 = alldat[subject_index][1]

dict_keys(['t_off', 'stim_id', 't_on', 'srate', 'V', 'scale_uv', 'locs', 'hemisphere', 'lobe', 'gyrus', 'Brodmann_Area'])
dict_keys(['stim_id', 'stim_cat', 'stim_noise', 't_on', 't_off', 'key_press', 'V', 'categories', 'scale_uv', 'locs', 'hemisphere', 'lobe', 'gyrus', 'Brodmann_Area'])


In [ ]:
# @title Get broadband
from scipy import signal

V = dat1['V'].astype('float32')

# Remove slower
b, a = signal.butter(3, [50], btype='high', fs=1000)
V = signal.filtfilt(b, a, V, 0)

# power
V = np.abs(V)**2

# Remove fast variations
b, a = signal.butter(3, [10], btype='low', fs=1000)
V = signal.filtfilt(b, a, V, 0)

# Normalize to 0
V = V/V.mean(0)

In [ ]:
'''
Slice voltages in stimulus time windows
'''

# 1. Define your parameters
chunk_size = 800
stim_V = V[dat1['t_on'][0]:]
num_channels = stim_V.shape[1]

# Set time window
# Probably need to be refactored
chunk_size = 800
num_chunks = len(stim_V) // chunk_size
truncated_length = num_chunks * chunk_size
stim_V_truncated = stim_V[:truncated_length]

# 3. Slice voltage in epochs
tim_V_3d = stim_V_truncated.reshape(num_chunks, chunk_size, num_channels)

# 4. Get mean across all ectrodes per epoch
stim_values = np.mean(stim_V_3d, axis=1)
